# 04 - QLoRA Fine-Tuning Diagnostic (v4)

Thin Kaggle runner for the `v4_diagnostic` smoke run. It verifies the v4 training stack (diverse output formats, expanded implications) before the full `v4_improved` run.

Run full training only after this diagnostic notebook passes and artifacts are inspected.

In [ ]:
!pip install -q -U datasets transformers trl peft accelerate bitsandbytes pyyaml
!pip install -q -U unsloth_zoo
!pip install -q -U --no-deps unsloth

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/W-Kaski/fingpt-qlora.git"
WORKDIR = Path("/kaggle/working")
REPO_DIR = WORKDIR / "fingpt-qlora"
CONFIG = "configs/experiments/v4_improved.yaml"

os.chdir(WORKDIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir(REPO_DIR)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Repo commit: {commit}")

In [ ]:
# Validate local control-plane code before spending GPU time.
!python -m unittest discover tests
!python -m compileall -q src tests


In [ ]:
# Build or refresh data splits.
!python -m src.data.download
!python -m src.data.preprocess
!python -m src.data.format_chat
!python -m src.data.merge_datasets
!python -m src.data.splits
!python -m src.data.manifest --splits-dir data/splits --output results/data_manifest_v3.json


In [ ]:
import json

with open("data/splits/sharegpt_all.json") as f:
    all_data = json.load(f)

# Check output format diversity (v4 has 5 variants)
samples = [ex["conversations"][-1]["value"] for ex in all_data[:100]]
has_sentiment_header = sum(1 for s in samples if "## Sentiment Analysis" in s)
has_bold_label = sum(1 for s in samples if "**Positive**" in s or "**Negative**" in s or "**Neutral**" in s)
has_driver = sum(1 for s in samples if "The primary driver is" in s)
has_concise = sum(1 for s in samples if s.startswith("**"))

print(f"Sample of {len(samples)} outputs:")
print(f"  Classic header (## Sentiment Analysis): {has_sentiment_header}")
print(f"  Bold label variant: {has_bold_label}")
print(f"  Driver-first variant: {has_driver}")
print(f"  Concise variant: {has_concise}")
print(f"\nFirst 3 samples:")
for i, s in enumerate(samples[:3]):
    print(f"  [{i}] {s[:120]}...")

In [ ]:
# Train LoRA adapter. All hyperparameters come from CONFIG.
!python -m src.train.train_sft --config {CONFIG} --output-suffix diagnostic


In [ ]:
from pathlib import Path

for artifact in [
    "results/data_manifest_v3.json",
    "outputs/v4_improved-kaggle/run_manifest.json",
    "outputs/v4_improved-kaggle/trainer_state.json",
    "outputs/v4_improved-kaggle/config_used.yaml",
]:
    path = Path(artifact)
    print(f"{artifact}: {path.exists()}")
    assert path.exists(), artifact